# Main Model

## Overview

The Main Model serves as the central coordinator of the multi-agent system. It interprets user requests, determines the appropriate workflow, and delegates tasks to the Architecture, Builder, and Review models in the correct sequence.

Rather than producing code or performing analysis itself, the Main Model manages the entire development pipeline — routing information, requesting files when needed, and ensuring that each specialized agent receives the right context and tasks. It is the only agent that communicates directly with the user, providing clear, structured responses while maintaining strict control over the multi-step workflow.

## Example Usage

### System Instruction

In [7]:
from system_instruction import getMainSystemInstruction

system_instructon = getMainSystemInstruction()

In [ ]:

from openai import OpenAI
import json
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

def call_agent(system_prompt, user_message, model="openai/gpt-oss-20b:free"):
    """Call any orchestrator agent with system + user messages."""
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return resp.choices[0].message.content


### Flow 1: User → Main → Architecture → ...


In [ ]:
user_request = "Create a VS Code extension that highlights TODO comments."

main_output_1 = call_agent(
    system_instructon,
    json.dumps({"user_request": user_request})
)

print(main_output_1)

Main Agent Output to Architecture:
{
  "target": "architecture",
  "context": "User wants to create a VS Code extension that highlights TODO comments in the editor.",
  "tasks": [
    "Define the overall architecture for the extension, including activation events, configuration, command registration, and highlighting mechanism.",
    "Outline the file structure and key files (e.g., extension.ts, package.json, README.md, test files).",
    "Specify dependencies (e.g., vscode API, any libraries for syntax parsing).",
    "Recommend a testing strategy (unit tests for highlighting logic, integration tests for VS Code integration)."
  ]
}


### Flow 2: User → Main → User


In [ ]:
user_request2 = "How to install npm from cli?"

main_output_2 = call_agent(
    system_instructon,
    json.dumps({"user_request": user_request2})
)

print(main_output_2)


Main Agent Output to Architecture:
{
    "target": "user",
    "response": "You can install npm by installing Node.js, which includes npm. For most systems, run a package manager command:\n- **macOS (Homebrew):** `brew install node`\n- **Ubuntu/Debian:** `sudo apt-get update && sudo apt-get install -y nodejs npm`\n- **Windows (Chocolatey):** `choco install nodejs`\nAfter installation, verify by running `node -v` and `npm -v` in your terminal."
}


### Flow 3: User → Main → Reviewer → ...


In [8]:
user_request3 = """
Is this code correct? 
int main(){
    return a + 7;
}
"""

main_output_3 = call_agent(
    system_instructon,
    json.dumps({"user_request": user_request3})
)

print(main_output_3)

{
    "target": "reviewer",
    "context": "User requests a review of the following code snippet for correctness: `int main(){ return a + 7; }`",
    "tasks": ["Analyze the code for syntactic and semantic correctness, identify missing declarations or includes, and provide a concise explanation of any errors or concerns."]
}
